# PromptMap Run Metrics and Comparison

This notebook loads PromptMap result JSON files from `results/`, builds run/test/iteration level metric tables, compares runs, and writes chart images to `graphs/generated/`.

## 1. Setup

In [10]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

ROOT = Path.cwd()
if not (ROOT / 'results').exists() and (ROOT.parent / 'results').exists():
    ROOT = ROOT.parent

RESULTS_DIR = ROOT / 'results'
GRAPHS_DIR = ROOT / 'graphs'
OUTPUT_DIR = GRAPHS_DIR / 'generated'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {ROOT}')
print(f'Results directory: {RESULTS_DIR}')
print(f'Chart output directory: {OUTPUT_DIR}')

Project root: d:\BDS\BTP\promptmap
Results directory: d:\BDS\BTP\promptmap\results
Chart output directory: d:\BDS\BTP\promptmap\graphs\generated


## 2. Load Runs

In [11]:
def load_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

def list_run_files(results_dir: Path):
    return sorted(
        p for p in results_dir.glob('*.json')
        if p.name != 'runs_index.json' and p.is_file()
    )

run_files = list_run_files(RESULTS_DIR)
runs = []
for path in run_files:
    try:
        data = load_json(path)
        data['_path'] = str(path)
        data['_run_id'] = path.stem
        runs.append(data)
    except Exception as exc:
        print(f'Skipping {path.name}: {exc}')

print(f'Loaded {len(runs)} run file(s).')
print([r['_run_id'] for r in runs])

Loaded 5 run file(s).
['2701c8eb', '27ecf231', '6224a899', 'a7a1e416', 'c6c6b4f1']


## 3. Build Metric Tables

In [14]:
def as_pct(value):
    if value is None or value == '':
        return 0.0
    try:
        if math.isinf(value):
            return np.nan
        return float(value)
    except Exception:
        return 0.0

def parse_pass_rate(value):
    if isinstance(value, str) and '/' in value:
        left, right = value.split('/', 1)
        try:
            left, right = float(left), float(right)
            return round(left / right * 100, 1) if right else np.nan
        except Exception:
            return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    return np.nan

def judge_label(judges):
    if not judges:
        return 'none'
    return ', '.join(j.get('model', '') for j in judges)

run_rows = []
test_rows = []
iteration_rows = []
type_rows = []
severity_rows = []

for run in runs:
    run_id = run['_run_id']
    print(run_id)
    meta = run.get('meta', {})
    config = meta.get('config', {}) if isinstance(meta.get('config'), dict) else {}
    tests = run.get('tests', {})
    judges = meta.get('judge_models') or config.get('judges') or []
    prompt_path = meta.get('system_prompts_path') or config.get('prompts_path') or ''

    run_rows.append({
        'run_id': run_id,
        'status': meta.get('status', ''),
        'target_model': meta.get('target_model', config.get('target_model', '')),
        'target_model_type': meta.get('target_model_type', config.get('target_model_type', '')),
        'system_prompt_file': Path(prompt_path).name if prompt_path else '',
        'started_at': meta.get('started_at', ''),
        'finished_at': meta.get('finished_at', ''),
        'iterations_per_test': meta.get('iterations_per_test', config.get('iterations', np.nan)),
        'total_tests': meta.get('total_tests', len(tests)),
        'judged_tests': meta.get('judged_tests', 0),
        'pending_tests': meta.get('pending_tests', 0),
        'passed': meta.get('total_passed', 0),
        'failed': meta.get('total_failed', 0),
        'pass_rate_pct': as_pct(meta.get('pass_rate_pct', 0)),
        'attack_success_rate_pct': as_pct(meta.get('attack_success_rate_pct', 0)),
        'judge_count': len(judges),
        'judge_models': judge_label(judges),
        'judged_iterations': meta.get('judged_iterations', 0),
        'judge_disagreement_rate_pct': as_pct(meta.get('judge_disagreement_rate_pct', 0)),
        'judge_agreement_rate_avg': as_pct(meta.get('judge_agreement_rate_avg', 0)),
        'judge_unanimous_rate_pct': as_pct(meta.get('judge_unanimous_rate_pct', 0)),
        'flakiness_rate_pct': as_pct(meta.get('flakiness_rate_pct', 0)),
        'critical_failure_rate_pct': as_pct(meta.get('critical_failure_rate_pct', 0)),
        'deterministic_failure_count': meta.get('deterministic_failure_count', 0),
        'judge_failure_count': meta.get('judge_failure_count', 0),
        'prompt_steal_rate_pct': as_pct(meta.get('prompt_steal_rate_pct', 0)),
        'file_path': run.get('_path', ''),
    })

    for attack_type, stats in (meta.get('type_stats') or {}).items():
        type_rows.append({'run_id': run_id, 'attack_type': attack_type, **stats})

    for severity, stats in (meta.get('sev_stats') or {}).items():
        severity_rows.append({'run_id': run_id, 'severity': severity, **stats})

    for test_name, test in tests.items():
        iterations = test.get('iterations', [])
        test_rows.append({
            'run_id': run_id,
            'test_name': test_name,
            'attack_type': test.get('type', ''),
            'severity': test.get('severity', ''),
            'passed': test.get('passed'),
            'judged': test.get('judged', False),
            'pass_rate': test.get('pass_rate'),
            'pass_rate_pct': parse_pass_rate(test.get('pass_rate')),
            'passed_count': test.get('passed_count', 0),
            'total_iterations': test.get('total_iterations', len(iterations)),
            'prompt_preview': str(test.get('prompt', ''))[:160],
        })

        for iteration in iterations:
            votes = iteration.get('judge_votes') or []
            vote_verdicts = [v.get('verdict') for v in votes]
            deterministic_findings = iteration.get('deterministic_findings') or []
            iteration_rows.append({
                'run_id': run_id,
                'test_name': test_name,
                'attack_type': test.get('type', ''),
                'severity': test.get('severity', ''),
                'iteration': iteration.get('iteration'),
                'is_error': iteration.get('is_error', False),
                'final_verdict': iteration.get('final_verdict') or iteration.get('verdict'),
                'judge_pass_votes': vote_verdicts.count('pass'),
                'judge_fail_votes': vote_verdicts.count('fail'),
                'judge_vote_count': len(votes),
                'judge_disagreement': iteration.get('judge_disagreement', False),
                'judge_agreement_rate': iteration.get('judge_agreement_rate', np.nan),
                'deterministic_failure': bool(deterministic_findings),
                'deterministic_finding_count': len(deterministic_findings),
                'response_chars': len(iteration.get('llm_response') or ''),
                'reason': iteration.get('reason', ''),
            })

runs_df = pd.DataFrame(run_rows)
tests_df = pd.DataFrame(test_rows)
iterations_df = pd.DataFrame(iteration_rows)
type_df = pd.DataFrame(type_rows)
severity_df = pd.DataFrame(severity_rows)

for df in [runs_df, tests_df, iterations_df, type_df, severity_df]:
    if not df.empty:
        df.columns = [c.strip() for c in df.columns]

print('runs_df:', runs_df.shape)
print('tests_df:', tests_df.shape)
print('iterations_df:', iterations_df.shape)
print('type_df:', type_df.shape)
print('severity_df:', severity_df.shape)

2701c8eb
27ecf231
6224a899
a7a1e416
c6c6b4f1
runs_df: (5, 27)
tests_df: (9, 11)
iterations_df: (26, 16)
type_df: (2, 6)
severity_df: (2, 7)


## 4. Run Summary Tables

In [15]:
summary_cols = [
    'run_id', 'status', 'target_model', 'system_prompt_file', 'total_tests',
    'judged_tests', 'passed', 'failed', 'pass_rate_pct', 'attack_success_rate_pct',
    'judge_count', 'judge_disagreement_rate_pct', 'flakiness_rate_pct',
    'critical_failure_rate_pct'
]

if runs_df.empty:
    print('No run JSON files found.')
else:
    display(runs_df[summary_cols].sort_values(['status', 'started_at', 'run_id'], ascending=[True, False, True]))

KeyError: 'started_at'

In [ ]:
if not tests_df.empty:
    failing_tests = tests_df[(tests_df['judged'] == True) & (tests_df['passed'] == False)].copy()
    failing_tests = failing_tests.sort_values(['run_id', 'severity', 'attack_type', 'test_name'])
    display(failing_tests[['run_id', 'test_name', 'attack_type', 'severity', 'pass_rate', 'prompt_preview']].head(25))
else:
    print('No tests found.')

## 5. Chart Helpers

In [ ]:
def savefig(name):
    path = OUTPUT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches='tight')
    print(f'Saved {path}')

def label_bars(ax, fmt='{:.1f}'):
    for container in ax.containers:
        labels = []
        for value in container.datavalues:
            if pd.isna(value):
                labels.append('')
            elif abs(value - round(value)) < 1e-9:
                labels.append(str(int(round(value))))
            else:
                labels.append(fmt.format(value))
        ax.bar_label(container, labels=labels, fontsize=8, padding=2)

def short_label(series):
    return series.astype(str).str.replace(':latest', '', regex=False).str.replace('openchat:7b-v3.5-q4_0', 'openchat-q4', regex=False)

## 6. Run-Level Comparison Graphs

In [ ]:
if runs_df.empty:
    print('No runs to plot.')
else:
    plot_df = runs_df.copy()
    plot_df['label'] = plot_df['run_id'] + '\n' + short_label(plot_df['target_model'])
    plot_df = plot_df.sort_values('started_at')

    fig, ax = plt.subplots(figsize=(max(8, len(plot_df) * 1.2), 4.8))
    x = np.arange(len(plot_df))
    width = 0.38
    ax.bar(x - width/2, plot_df['pass_rate_pct'], width, label='Pass rate %', color='#2f7d5c')
    ax.bar(x + width/2, plot_df['attack_success_rate_pct'], width, label='Attack success %', color='#b84a4a')
    ax.set_title('Run Security Outcome Comparison')
    ax.set_ylabel('Percent')
    ax.set_ylim(0, 105)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['label'], rotation=35, ha='right')
    ax.legend()
    label_bars(ax)
    savefig('run_pass_vs_attack_success.png')
    plt.show()

In [ ]:
if not runs_df.empty:
    counts = runs_df['status'].fillna('unknown').value_counts()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(counts.index, counts.values, color=['#4c78a8', '#f58518', '#54a24b', '#e45756'][:len(counts)])
    ax.set_title('Run Status Counts')
    ax.set_ylabel('Runs')
    label_bars(ax, fmt='{:.0f}')
    savefig('run_status_counts.png')
    plt.show()

In [ ]:
metric_cols = [
    'judge_disagreement_rate_pct', 'judge_unanimous_rate_pct', 'flakiness_rate_pct',
    'critical_failure_rate_pct', 'prompt_steal_rate_pct'
]

judged_plot_df = runs_df[runs_df['judged_tests'] > 0].copy() if not runs_df.empty else pd.DataFrame()
if judged_plot_df.empty:
    print('No judged runs yet. Judge at least one run to plot judge/flakiness metrics.')
else:
    judged_plot_df['label'] = judged_plot_df['run_id'] + '\n' + short_label(judged_plot_df['target_model'])
    metric_df = judged_plot_df.set_index('label')[metric_cols]
    ax = metric_df.plot(kind='bar', figsize=(max(9, len(metric_df) * 1.3), 5.2), width=0.82)
    ax.set_title('Judged Run Quality Metrics')
    ax.set_ylabel('Percent')
    ax.set_ylim(0, 105)
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    plt.xticks(rotation=35, ha='right')
    savefig('judged_run_quality_metrics.png')
    plt.show()

## 7. Attack Type and Severity Comparisons

In [ ]:
if type_df.empty:
    print('No type_stats found. These appear after runs are judged or summary stats are recomputed.')
else:
    type_df['pass_rate_pct'] = type_df['pass_rate_pct'].map(as_pct)
    pivot = type_df.pivot_table(index='attack_type', columns='run_id', values='pass_rate_pct', aggfunc='mean')
    fig, ax = plt.subplots(figsize=(max(7, pivot.shape[1] * 1.2), max(4, pivot.shape[0] * 0.6)))
    im = ax.imshow(pivot.fillna(np.nan), aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
    ax.set_title('Pass Rate by Attack Type and Run')
    ax.set_xticks(np.arange(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns, rotation=35, ha='right')
    ax.set_yticks(np.arange(pivot.shape[0]))
    ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.iloc[i, j]
            text = '' if pd.isna(val) else f'{val:.0f}'
            ax.text(j, i, text, ha='center', va='center', color='black', fontsize=8)
    fig.colorbar(im, ax=ax, label='Pass rate %')
    savefig('attack_type_pass_rate_heatmap.png')
    plt.show()

In [ ]:
if severity_df.empty:
    print('No severity stats found.')
else:
    sev = severity_df.copy()
    sev['failed'] = pd.to_numeric(sev.get('failed', 0), errors='coerce').fillna(0)
    sev['passed'] = pd.to_numeric(sev.get('passed', 0), errors='coerce').fillna(0)
    order = ['low', 'medium', 'high', 'critical']
    severity_totals = sev.groupby('severity')[['passed', 'failed']].sum().reindex(order).dropna(how='all').fillna(0)
    ax = severity_totals.plot(kind='bar', stacked=True, figsize=(7, 4.5), color=['#2f7d5c', '#b84a4a'])
    ax.set_title('Total Outcomes by Severity')
    ax.set_ylabel('Tests')
    ax.set_xlabel('Severity')
    label_bars(ax, fmt='{:.0f}')
    savefig('severity_pass_fail_totals.png')
    plt.show()

## 8. Iteration-Level Diagnostics

In [ ]:
if iterations_df.empty:
    print('No iterations found.')
else:
    verdict_counts = iterations_df.assign(final_verdict=iterations_df['final_verdict'].fillna('pending'))
    verdict_counts = verdict_counts.groupby(['run_id', 'final_verdict']).size().unstack(fill_value=0)
    ax = verdict_counts.plot(kind='bar', stacked=True, figsize=(max(8, len(verdict_counts) * 1.1), 4.8), color={'pass': '#2f7d5c', 'fail': '#b84a4a', 'pending': '#8a8f98'})
    ax.set_title('Iteration Verdict Counts by Run')
    ax.set_ylabel('Iterations')
    ax.set_xlabel('Run')
    plt.xticks(rotation=35, ha='right')
    savefig('iteration_verdict_counts.png')
    plt.show()

In [ ]:
if iterations_df.empty:
    print('No iterations found.')
else:
    diag = iterations_df.groupby('run_id').agg(
        deterministic_failures=('deterministic_failure', 'sum'),
        judge_disagreements=('judge_disagreement', 'sum'),
        avg_response_chars=('response_chars', 'mean'),
    ).reset_index()
    display(diag)

    ax = diag.set_index('run_id')[['deterministic_failures', 'judge_disagreements']].plot(kind='bar', figsize=(max(7, len(diag) * 1.1), 4.5), color=['#6f4e7c', '#d98c3a'])
    ax.set_title('Deterministic Findings and Judge Disagreements')
    ax.set_ylabel('Iteration count')
    ax.set_xlabel('Run')
    plt.xticks(rotation=35, ha='right')
    label_bars(ax, fmt='{:.0f}')
    savefig('deterministic_and_disagreement_counts.png')
    plt.show()

## 9. Export Tables

In [ ]:
exports = {
    'runs_metrics.csv': runs_df,
    'tests_metrics.csv': tests_df,
    'iterations_metrics.csv': iterations_df,
    'type_metrics.csv': type_df,
    'severity_metrics.csv': severity_df,
}

for filename, df in exports.items():
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False)
    print(f'Wrote {path} ({len(df)} rows)')

## 10. Quick Interpretation

- `pass_rate_pct`: share of judged tests that resisted the attack.
- `attack_success_rate_pct`: inverse security view; higher means more attacks succeeded.
- `judge_disagreement_rate_pct`: how often judge models disagreed at iteration level.
- `flakiness_rate_pct`: tests with mixed pass/fail outcomes across iterations.
- `critical_failure_rate_pct`: high-severity failures among judged tests.
- `deterministic_failure_count`: failures caught by programmatic checks rather than only judge votes.